# How `bindcurve` uses the four-state models

This notebook is intentionally implementation-oriented. Instead of re-deriving the full Roehrl algebra symbolically, it shows the runtime workflow used by `bindcurve` for the incomplete-competition models `comp_4st_specific` and `comp_4st_total`.

**Reference**

Roehrl, M. H. A.; Wang, J. Y.; Wagner, G. *Biochemistry* **2004**, 43(51), 16056-16066. DOI: [10.1021/bi048233g](https://doi.org/10.1021/bi048233g)

The key detail is that `bindcurve` does **not** solve the four-state model directly in literal free receptor concentration. It solves a quintic in the transformed coordinate

$$
r_{\mathrm{equiv}} = K_{ds}\frac{FSB}{1-FSB},
$$

and then converts the selected physical root back to tracer-bound fraction with

$$
FSB = \frac{r_{\mathrm{equiv}}}{K_{ds} + r_{\mathrm{equiv}}}.
$$

That recovered `FSB` is what gets mapped to the experimental signal.


## When the four-state models are used

The four-state models are for **incomplete competition**. In addition to `RLs` and `RL`, the system allows the ternary complex `RLLs`, so the tracer can still bind after the competitor is already present.

- `Kds`: tracer affinity for free receptor
- `Kd`: competitor affinity for free receptor
- `Kd3`: tracer affinity for the receptor-competitor complex
- `comp_4st_specific`: solves the specific-binding model
- `comp_4st_total`: uses the same machinery, but replaces `Kd` by `(1 + N) * Kd`

The implementation lives in `bindcurve/modeling/competitive.py`.


In [4]:
from pprint import pprint

import numpy as np
import pandas as pd

from bindcurve.modeling.competitive import (
    CompetitiveFourStateSpecificKdModel,
    CompetitiveFourStateTotalKdModel,
    _competitive_four_state_coefficients,
    _direct_bound_fraction,
    _equivalent_receptor_bounds,
    _equivalent_receptor_from_bound_fraction,
    _select_physical_root,
)

np.set_printoptions(precision=6, suppress=True)

params = {
    "RT": 0.05,
    "LsT": 0.005,
    "Kds": 0.02,
    "Kd": 1.6,
    "Kd3": 0.5,
    "N": 0.35,
    "ymin": 5.0,
    "ymax": 95.0,
}

print("Example parameter set used below:")
pprint(params)
print(f"\nEffective Kd for comp_4st_total: {(1 + params['N']) * params['Kd']:.6f} uM")


Example parameter set used below:
{'Kd': 1.6,
 'Kd3': 0.5,
 'Kds': 0.02,
 'LsT': 0.005,
 'N': 0.35,
 'RT': 0.05,
 'ymax': 95.0,
 'ymin': 5.0}

Effective Kd for comp_4st_total: 2.160000 uM


## The transformed coordinate and its physical interval

In the three-state model, the equivalent coordinate collapses to literal free receptor. In the four-state model it generally does **not**. The physically valid interval for `r_equiv` is determined by two endpoint bound fractions:

- the direct-binding limit at `LT = 0`, governed by `Kds`
- the asymptotic high-competitor limit, governed by `Kd3`

Those two endpoint fractions are converted into `r_equiv`, and that interval is used to filter the numeric roots of the quintic.


In [5]:
initial_fraction = _direct_bound_fraction(
    Kd=params["Kds"],
    RT=params["RT"],
    LsT=params["LsT"],
)
asymptotic_fraction = _direct_bound_fraction(
    Kd=params["Kd3"],
    RT=params["RT"],
    LsT=params["LsT"],
)

initial_r_equiv = _equivalent_receptor_from_bound_fraction(
    initial_fraction,
    Kds=params["Kds"],
)
asymptotic_r_equiv = _equivalent_receptor_from_bound_fraction(
    asymptotic_fraction,
    Kds=params["Kds"],
)
lower_bound, upper_bound = _equivalent_receptor_bounds(
    RT=params["RT"],
    LsT=params["LsT"],
    Kds=params["Kds"],
    Kd3=params["Kd3"],
)

print(f"Tracer-bound fraction at LT = 0: {initial_fraction:.6f}")
print(f"Asymptotic tracer-bound fraction at very large LT: {asymptotic_fraction:.6f}")
print(f"r_equiv at LT = 0: {initial_r_equiv:.6f}")
print(f"Asymptotic r_equiv: {asymptotic_r_equiv:.6f}")
print(f"Feasible interval used for root selection: [{lower_bound:.6f}, {upper_bound:.6f}]")
print(f"RT for comparison: {params['RT']:.6f}")


Tracer-bound fraction at LT = 0: 0.699265
Asymptotic tracer-bound fraction at very large LT: 0.090163
r_equiv at LT = 0: 0.046504
Asymptotic r_equiv: 0.001982
Feasible interval used for root selection: [0.001982, 0.046504]
RT for comparison: 0.050000


## One competitor concentration: build the quintic and pick the physical root

For each competitor concentration `LT`, `bindcurve` builds a quintic in `r_equiv`:

$$
a r_{\mathrm{equiv}}^5 + b r_{\mathrm{equiv}}^4 + c r_{\mathrm{equiv}}^3 + d r_{\mathrm{equiv}}^2 + e r_{\mathrm{equiv}} + f = 0.
$$

The coefficients are the closed-form expressions stored in `competitive.py`. The solver finds all numeric roots, rejects nonphysical ones, and keeps the feasible root with the smallest scaled residual.


In [6]:
LT = 0.1
coefficients = _competitive_four_state_coefficients(
    LT,
    RT=params["RT"],
    LsT=params["LsT"],
    Kds=params["Kds"],
    Kd=params["Kd"],
    Kd3=params["Kd3"],
)

print(f"Quintic coefficients at LT = {LT} uM:")
for name, coefficient in zip("abcdef", coefficients, strict=True):
    print(f"  {name} = {coefficient:.12g}")

roots = np.roots(coefficients)
print("\nAll roots:")
for root in roots:
    sign = "+" if np.imag(root) >= 0 else "-"
    print(f"  {np.real(root): .12g} {sign} {abs(np.imag(root)):.12g}j")

selected_root = _select_physical_root(
    coefficients,
    lower_bound=lower_bound,
    upper_bound=upper_bound,
)
fraction_bound = selected_root / (params["Kds"] + selected_root)

print(f"\nSelected physical root: r_equiv = {selected_root:.12g}")
print(f"Recovered tracer-bound fraction: FSB = {fraction_bound:.12g}")
print(f"Polynomial residual at selected root: {np.polyval(coefficients, selected_root):.3e}")


Quintic coefficients at LT = 0.1 uM:
  a = -0.0001
  b = -0.00016454
  c = -2.699484e-06
  d = 2.400551e-07
  e = 7.656216e-09
  f = 6.15128e-11

All roots:
  -1.62792952837 + 0j
   0.0439994824223 + 0j
  -0.0214754533465 + 0j
  -0.0200203685679 + 0j
  -0.0199741321382 + 0j

Selected physical root: r_equiv = 0.0439994824223
Recovered tracer-bound fraction: FSB = 0.687497472745
Polynomial residual at selected root: 1.965e-24


## End-to-end model evaluation

Once the physical `r_equiv` root is selected, `bindcurve` converts it to `FSB` and then to signal:

$$
y = ymin + (ymax - ymin)FSB.
$$

The total-binding model does not use a separate polynomial form. It simply evaluates the same four-state machinery with `Kd_eff = (1 + N) * Kd`.


In [7]:
concentration = np.logspace(-3, 2, 8)
specific_model = CompetitiveFourStateSpecificKdModel()
total_model = CompetitiveFourStateTotalKdModel()

specific = specific_model.evaluate(
    concentration,
    ymin=params["ymin"],
    ymax=params["ymax"],
    RT=params["RT"],
    LsT=params["LsT"],
    Kds=params["Kds"],
    Kd=params["Kd"],
    Kd3=params["Kd3"],
)
total_zero_n = total_model.evaluate(
    concentration,
    ymin=params["ymin"],
    ymax=params["ymax"],
    RT=params["RT"],
    LsT=params["LsT"],
    Kds=params["Kds"],
    Kd=params["Kd"],
    Kd3=params["Kd3"],
    N=0.0,
)
total = total_model.evaluate(
    concentration,
    ymin=params["ymin"],
    ymax=params["ymax"],
    RT=params["RT"],
    LsT=params["LsT"],
    Kds=params["Kds"],
    Kd=params["Kd"],
    Kd3=params["Kd3"],
    N=params["N"],
)

effective_kd = (1 + params["N"]) * params["Kd"]
specific_with_effective_kd = specific_model.evaluate(
    concentration,
    ymin=params["ymin"],
    ymax=params["ymax"],
    RT=params["RT"],
    LsT=params["LsT"],
    Kds=params["Kds"],
    Kd=effective_kd,
    Kd3=params["Kd3"],
)

table = pd.DataFrame(
    {
        "LT_uM": concentration,
        "specific": specific,
        "total_N0": total_zero_n,
        "total_with_N": total,
    }
)

print("Response table (percent signal):")
print(table.to_string(index=False, float_format=lambda value: f"{value:0.6f}"))
print()
print(f"max |specific - total(N=0)| = {np.max(np.abs(specific - total_zero_n)):.3e}")
print(f"Effective Kd used by comp_4st_total = {effective_kd:.6f} uM")
print(
    f"max |total(N={params['N']}) - specific(Kd_eff)| = "
    f"{np.max(np.abs(total - specific_with_effective_kd)):.3e}"
)


Response table (percent signal):
     LT_uM  specific  total_N0  total_with_N
  0.001000 67.923039 67.923039     67.925776
  0.005179 67.877994 67.877994     67.892154
  0.026827 67.645811 67.645811     67.718647
  0.138950 66.472860 66.472860     66.836925
  0.719686 61.110011 61.110011     62.686485
  3.727594 44.690967 44.690967     48.545368
 19.306977 24.571622 24.571622     27.505069
100.000000 15.785939 15.785939     16.658413

max |specific - total(N=0)| = 0.000e+00
Effective Kd used by comp_4st_total = 2.160000 uM
max |total(N=0.35) - specific(Kd_eff)| = 0.000e+00


## Takeaway

The practical four-state workflow in `bindcurve` is:

1. Build the quintic coefficients in `r_equiv` for each competitor concentration.
2. Solve the quintic numerically.
3. Keep the effectively real root inside the feasible transformed-coordinate interval.
4. Convert that root back to `FSB` with `FSB = r_equiv / (Kds + r_equiv)`.
5. Convert `FSB` to the observed response using `ymin` and `ymax`.
6. For `comp_4st_total`, use the same machinery with `Kd_eff = (1 + N) * Kd`.

So the important implementation choice is not the raw symbolic derivation itself, but the transformed coordinate, the physical root-selection rule, and the `Kd` scaling used by the total-binding variant.
